# 01 - Autograd Engine from Scratch

**Goal:** Build a minimal reverse-mode automatic differentiation engine. This is a real interview question at companies like Google, Meta, and startups.

**What we build:**
- A `Value` class that tracks computations in a DAG (directed acyclic graph)
- Reverse-mode autodiff (backpropagation) via topological sort
- Support for: `+`, `*`, `**`, `relu`, `tanh`, `exp`, `log`

---

### Why computational graphs?

Every neural network computation is a composition of elementary ops. If we record these ops in a graph, we can mechanically apply the chain rule to get gradients of *any* output w.r.t. *any* input.

**Key formula** (chain rule along a path):

$$\frac{\partial L}{\partial x} = \sum_{\text{paths } x \to L} \prod_{\text{edges on path}} \text{local gradient}$$

In practice we don't enumerate paths. Reverse-mode accumulates gradients via a backward pass through the topologically sorted graph.

### Forward-mode vs Reverse-mode

| | Forward-mode | Reverse-mode |
|---|---|---|
| Computes | $\frac{\partial \text{all outputs}}{\partial \text{one input}}$ | $\frac{\partial \text{one output}}{\partial \text{all inputs}}$ |
| Cost per pass | One pass per input dim | One pass per output dim |
| Best when | Few inputs, many outputs | Few outputs (loss), many inputs (params) |
| Used in | Dual numbers, JVPs | Backprop, VJPs |

Neural nets have scalar loss and millions of params, so reverse-mode wins. This is why backprop exists.

**Jacobian connection:** For $f: \mathbb{R}^n \to \mathbb{R}^m$, the Jacobian is $J \in \mathbb{R}^{m \times n}$. Forward-mode computes $Jv$ (Jacobian-vector product). Reverse-mode computes $v^T J$ (vector-Jacobian product). Backprop chains VJPs.

**Resources:**
- Baydin et al., "Automatic Differentiation in Machine Learning: a Survey" (2018)
- Karpathy's micrograd (the inspiration for this notebook)
- JAX docs on JVP vs VJP

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

## The `Value` Class

Each `Value` wraps a scalar and knows:
1. Its numeric value (`self.data`)
2. Its gradient (`self.grad`) -- accumulated during backward
3. Its children and the local backward function (`self._backward`)

The local backward function for an op encodes the chain rule for that op. For example:
- `c = a + b` => `da += dc`, `db += dc` (local grads are both 1)
- `c = a * b` => `da += b * dc`, `db += a * dc`
- `c = a ** n` => `da += n * a^(n-1) * dc`

In [ ]:
class Value:
    """A scalar value with automatic gradient computation."""
    
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None  # no-op by default
        self._prev = set(_children)
        self._op = _op
        self.label = label
    
    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out
    
    def __radd__(self, other):
        return self + other
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, other):
        return self + (-other)
    
    def __rsub__(self, other):
        return other + (-self)
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def __rmul__(self, other):
        return self * other
    
    def __truediv__(self, other):
        return self * (other ** -1)
    
    def __rtruediv__(self, other):
        return other * (self ** -1)
    
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only int/float powers supported"
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out
    
    def relu(self):
        out = Value(max(0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (1.0 if self.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out
    
    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1.0 - t**2) * out.grad
        out._backward = _backward
        return out
    
    def exp(self):
        e = math.exp(self.data)
        out = Value(e, (self,), 'exp')
        def _backward():
            self.grad += e * out.grad
        out._backward = _backward
        return out
    
    def log(self):
        out = Value(math.log(self.data), (self,), 'log')
        def _backward():
            self.grad += (1.0 / self.data) * out.grad
        out._backward = _backward
        return out
    
    def backward(self):
        """Reverse-mode autodiff. Topological sort then propagate gradients."""
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        self.grad = 1.0  # dL/dL = 1
        for node in reversed(topo):
            node._backward()

## Quick sanity check

Verify against known derivatives.

In [ ]:
# Test: d/dx (x^3) at x=2 should be 12
x = Value(2.0, label='x')
y = x ** 3
y.backward()
print(f"x^3 at x=2: value={y.data}, dx={x.grad} (expected 12.0)")
assert abs(x.grad - 12.0) < 1e-6

# Test: d/dx (relu(x)) at x=-1 should be 0, at x=2 should be 1
x1 = Value(-1.0)
y1 = x1.relu()
y1.backward()
print(f"relu(-1): value={y1.data}, dx={x1.grad} (expected 0.0)")

x2 = Value(2.0)
y2 = x2.relu()
y2.backward()
print(f"relu(2):  value={y2.data}, dx={x2.grad} (expected 1.0)")

# Test: d/dx tanh(x) at x=0 should be 1
x3 = Value(0.0)
y3 = x3.tanh()
y3.backward()
print(f"tanh(0):  value={y3.data:.4f}, dx={x3.grad:.4f} (expected 1.0)")

## Demo 1: Gradients of f(x, y) = x*y + sin(x)

We don't have a native `sin` op, so we'll implement it using the Taylor expansion or just add it as an op. Let's add it cleanly.

In [ ]:
def sin(v):
    """Sine operation for Value objects."""
    out = Value(math.sin(v.data), (v,), 'sin')
    def _backward():
        v.grad += math.cos(v.data) * out.grad
    out._backward = _backward
    return out

# f(x, y) = x*y + sin(x)
# df/dx = y + cos(x)
# df/dy = x

x = Value(2.0, label='x')
y = Value(3.0, label='y')
f = x * y + sin(x)
f.backward()

print(f"f(2, 3) = 2*3 + sin(2) = {f.data:.6f}")
print(f"df/dx = y + cos(x) = 3 + cos(2) = {3 + math.cos(2):.6f}")
print(f"Autograd df/dx = {x.grad:.6f}")
print(f"df/dy = x = 2.0")
print(f"Autograd df/dy = {y.grad:.6f}")

assert abs(x.grad - (3 + math.cos(2))) < 1e-6
assert abs(y.grad - 2.0) < 1e-6
print("\nAll gradients match analytic solutions.")

## Numerical gradient check

Always validate autograd with finite differences: $\frac{\partial f}{\partial x} \approx \frac{f(x+h) - f(x-h)}{2h}$

In [ ]:
def numerical_grad(fn, inputs, idx, h=1e-5):
    """Compute df/d(inputs[idx]) via central differences."""
    orig = inputs[idx]
    inputs[idx] = orig + h
    f_plus = fn(*inputs)
    inputs[idx] = orig - h
    f_minus = fn(*inputs)
    inputs[idx] = orig  # restore
    return (f_plus - f_minus) / (2 * h)

# Test on f(x,y) = x*y + sin(x)
def f_numpy(x, y):
    return x * y + np.sin(x)

inputs = [2.0, 3.0]
num_dx = numerical_grad(f_numpy, inputs, 0)
num_dy = numerical_grad(f_numpy, inputs, 1)
print(f"Numerical df/dx = {num_dx:.6f}  (autograd: {3 + math.cos(2):.6f})")
print(f"Numerical df/dy = {num_dy:.6f}  (autograd: 2.000000)")

## Demo 2: Train a Neuron

A single neuron: $y = \text{tanh}(w_1 x_1 + w_2 x_2 + b)$

We'll train it to fit a simple target using MSE loss and manual gradient descent.

In [ ]:
# Training data: 2D inputs -> scalar target
# Target function: sign(x1 + x2) mapped through tanh
np.random.seed(42)
X_data = np.random.randn(20, 2)
y_data = np.tanh(X_data[:, 0] * 1.5 - X_data[:, 1] * 0.8 + 0.3)

# Initialize weights
w1 = Value(np.random.randn() * 0.1, label='w1')
w2 = Value(np.random.randn() * 0.1, label='w2')
b = Value(0.0, label='b')

lr = 0.05
losses = []

for epoch in range(100):
    # Forward pass: compute total loss over all data points
    total_loss = Value(0.0)
    for i in range(len(X_data)):
        x1_val = Value(X_data[i, 0])
        x2_val = Value(X_data[i, 1])
        # Neuron output
        pred = (w1 * x1_val + w2 * x2_val + b).tanh()
        # MSE contribution
        diff = pred - Value(y_data[i])
        total_loss = total_loss + diff ** 2
    
    total_loss = total_loss * (1.0 / len(X_data))  # mean
    losses.append(total_loss.data)
    
    # Backward pass
    # Zero grads first
    w1.grad = 0.0
    w2.grad = 0.0
    b.grad = 0.0
    total_loss.backward()
    
    # Update
    w1.data -= lr * w1.grad
    w2.data -= lr * w2.grad
    b.data -= lr * b.grad
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Loss: {total_loss.data:.6f} | w1={w1.data:.4f} w2={w2.data:.4f} b={b.data:.4f}")

print(f"\nFinal: w1={w1.data:.4f}, w2={w2.data:.4f}, b={b.data:.4f}")
print(f"Target: w1=1.5, w2=-0.8, b=0.3")

In [ ]:
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Single Neuron Training with Custom Autograd')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Building a tiny MLP

To make this more useful, let's add `Neuron`, `Layer`, and `MLP` abstractions on top of `Value`.

In [ ]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(np.random.randn() * (2.0 / nin)**0.5) for _ in range(nin)]
        self.b = Value(0.0)
    
    def __call__(self, x):
        # w . x + b
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()
    
    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

# Train a small MLP on XOR-like data
np.random.seed(42)
model = MLP(2, [4, 4, 1])  # 2 inputs -> 4 -> 4 -> 1 output
print(f"Number of parameters: {len(model.parameters())}")

# XOR data
X_xor = [[0, 0], [0, 1], [1, 0], [1, 1]]
y_xor = [-1, 1, 1, -1]  # using -1/1 since we use tanh

losses = []
for epoch in range(200):
    # Forward
    preds = [model(x) for x in X_xor]
    loss = sum((p - Value(yt))**2 for p, yt in zip(preds, y_xor))
    losses.append(loss.data)
    
    # Backward
    for p in model.parameters():
        p.grad = 0.0
    loss.backward()
    
    # Update
    lr = 0.05
    for p in model.parameters():
        p.data -= lr * p.grad
    
    if epoch % 40 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.data:.6f}")

print("\nPredictions:")
for x, yt in zip(X_xor, y_xor):
    pred = model(x)
    print(f"  Input: {x} | Target: {yt:+d} | Pred: {pred.data:+.4f}")

In [ ]:
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('MLP Training on XOR (Custom Autograd)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Visualizing the Computational Graph

Let's trace a small computation and display the graph structure.

In [ ]:
def trace(root):
    """Build sets of all nodes and edges in the graph."""
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def print_graph(root, indent=0):
    """Print the computation graph as a tree (may repeat shared nodes)."""
    prefix = "  " * indent
    label = root.label if root.label else root._op
    print(f"{prefix}[{label}] data={root.data:.4f} grad={root.grad:.4f}")
    for child in root._prev:
        print_graph(child, indent + 1)

# Simple example
a = Value(2.0, label='a')
b = Value(3.0, label='b')
c = a * b; c.label = 'c=a*b'
d = sin(a); d.label = 'd=sin(a)'
e = c + d; e.label = 'e=c+d'
e.backward()

print("Computation graph for f(a,b) = a*b + sin(a):")
print_graph(e)
print(f"\nNodes in graph: {len(trace(e)[0])}, Edges: {len(trace(e)[1])}")

## Gradient accumulation: a subtle point

When a value is used multiple times in the graph, its gradient must be **accumulated** (summed), not overwritten. This is the multivariate chain rule:

$$\frac{\partial L}{\partial x} = \sum_i \frac{\partial L}{\partial z_i} \frac{\partial z_i}{\partial x}$$

where $z_i$ are all the nodes that directly use $x$. Our implementation handles this with `+=` in the backward functions.

In [ ]:
# x is used twice: f = x*x = x^2, so df/dx = 2x
x = Value(3.0, label='x')
f = x * x  # x appears in both children of *
f.backward()
print(f"f = x*x at x=3: value={f.data}, grad={x.grad} (expected 6.0)")
assert abs(x.grad - 6.0) < 1e-6

# Another test: f = x + x = 2x, df/dx = 2
x = Value(5.0, label='x')
f = x + x
f.backward()
print(f"f = x+x at x=5: value={f.data}, grad={x.grad} (expected 2.0)")
assert abs(x.grad - 2.0) < 1e-6

# f = x*x + x (x used three times)
# df/dx = 2x + 1 = 7 at x=3
x = Value(3.0, label='x')
f = x * x + x
f.backward()
print(f"f = x*x+x at x=3: value={f.data}, grad={x.grad} (expected 7.0)")
assert abs(x.grad - 7.0) < 1e-6

## Comparison with PyTorch

If you have PyTorch installed, you can verify our engine matches it exactly.

In [ ]:
try:
    import torch
    
    # Our engine
    a = Value(2.0)
    b = Value(3.0)
    c = a * b + sin(a)
    c.backward()
    our_da, our_db = a.grad, b.grad
    
    # PyTorch
    at = torch.tensor(2.0, requires_grad=True)
    bt = torch.tensor(3.0, requires_grad=True)
    ct = at * bt + torch.sin(at)
    ct.backward()
    pt_da, pt_db = at.grad.item(), bt.grad.item()
    
    print(f"Our engine:  da={our_da:.6f}, db={our_db:.6f}")
    print(f"PyTorch:     da={pt_da:.6f}, db={pt_db:.6f}")
    print(f"Match: {abs(our_da - pt_da) < 1e-6 and abs(our_db - pt_db) < 1e-6}")
except ImportError:
    print("PyTorch not available. That's fine -- our numerical checks above confirm correctness.")

---

## Interview Notes

**"Implement autograd from scratch"** -- This comes up at Google Brain, DeepMind, and ML-focused startups.

Key points to hit:
1. **Computational graph as a DAG** -- each op creates a node with pointers to its inputs
2. **Topological sort** -- process nodes in reverse topological order for backward pass
3. **Gradient accumulation** -- `+=` not `=`, because a variable can be used multiple times (multivariate chain rule)
4. **Local backward functions** -- each op stores a closure that knows how to propagate gradients to its inputs
5. **Memory**: reverse-mode requires storing all intermediate values (forward pass) for the backward pass. This is the memory cost of backprop.

**Follow-up questions:**
- "How would you handle in-place operations?" (You can't easily -- they break the graph. PyTorch detects and raises errors.)
- "How does gradient checkpointing work?" (Trade compute for memory: don't store some intermediates, recompute them during backward.)
- "What about higher-order derivatives?" (Make the backward pass itself differentiable -- the grad values become part of a new graph.)

**Complexity:** For a graph with $N$ nodes and $E$ edges, backward is $O(N + E)$ -- same as forward. One backward pass gives you ALL gradients.

**Further reading:**
- PyTorch autograd internals: https://pytorch.org/docs/stable/notes/autograd.html
- JAX autodiff cookbook: https://jax.readthedocs.io/en/latest/notebooks/autodiff_cookbook.html